In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# 1. Create a URL object (no manual encoding needed)
url_object = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password="root",  # Ensure this is correct
    host="localhost",
    port=5432,
    database="BankingTransactionAnalysis(NovaBank)",
)

# 2. Create engine using the object
engine = create_engine(url_object)

# 3. Loading DFs
query = "SELECT * FROM branches"
branches_df = pd.read_sql(query, engine)
print(f"Rows loaded in branches_df: {branches_df.shape[0]}")

query = "SELECT * FROM customers"
customers_df = pd.read_sql(query, engine)
print(f"Rows loaded in customers_df: {customers_df.shape[0]}")

query = "SELECT * FROM accounts"
accounts_df = pd.read_sql(query, engine)
print(f"Rows loaded in accounts_df: {accounts_df.shape[0]}")

query = "SELECT * FROM transactions"
transactions_df = pd.read_sql(query, engine)
print(f"Rows loaded in transactions_df: {transactions_df.shape[0]}")

Rows loaded in branches_df: 10
Rows loaded in customers_df: 5000
Rows loaded in accounts_df: 6356
Rows loaded in transactions_df: 297129


In [2]:
# Feature Engineering - Transactions Column

In [3]:
transactions_df['is_withdrawal'] = (transactions_df['transaction_type']=='Atm Withdrawal') | ((transactions_df['transaction_type']=='Withdrawal or Deposit') & (transactions_df['debit_or_credit']=='Debit')) 
transactions_df.groupby('is_withdrawal')['transaction_id'].count() # 30369 Transactions are withdrawal rest 266760 are non withdrawal 

is_withdrawal
False    266291
True      30838
Name: transaction_id, dtype: int64

In [4]:
# Feature Engineering - Accounts Column 

In [5]:
accounts_df['balance_segment'] = pd.qcut(accounts_df['current_balance'], 3, labels=["Low Balance","Medium Balance", "High Balance"])
accounts_df.head(10)

,account_number,customer_id,account_type,opened_date,current_balance,account_status,is_non_transacting_account,balance_segment
0,ACC0000012,CUST00010,Savings,2022-06-04,12218.69,Active,False,Low Balance
1,ACC0000013,CUST00011,Savings,2023-03-07,0.00,Active,False,Low Balance
2,ACC0000018,CUST00015,Savings,2022-02-07,5000.00,Active,False,Low Balance
3,ACC0000020,CUST00017,Savings,2022-03-20,22846.90,Active,False,Medium Balance
4,ACC0000025,CUST00022,Current,2023-10-28,1085843.36,Active,False,High Balance
5,ACC0000028,CUST00025,Savings,2022-06-01,21047.74,Frozen,False,Medium Balance
6,ACC0000029,CUST00026,Savings,2023-10-17,136240.76,Active,False,High Balance
7,ACC0000030,CUST00027,Savings,2023-02-12,5000.00,Active,False,Low Balance
8,ACC0000031,CUST00028,Savings,2023-10-15,500.00,Active,False,Low Balance
9,ACC0000035,CUST00031,Savings,2023-07-07,36831.66,Active,False,Medium Balance


In [6]:
# Feature Engineering - Customers Column - recency_days

In [7]:
# defining the maximum transaction date
max_transaction_date = pd.to_datetime(transactions_df['transaction_datetime'].max())

# Calculating and storing last transaction dates per customers
last_transactions = transactions_df.groupby('customer_id')['transaction_datetime'].max().reset_index()
last_transactions.columns = ['customer_id', 'last_transaction_date']

# Merging customers with there last_transaction_date
customers_df = customers_df.merge(last_transactions, on='customer_id', how='left')

# Calculating days since last transaction
customers_df['days_since_last_trans'] = (max_transaction_date - customers_df['last_transaction_date']).dt.days
customers_df['days_since_last_trans'].isna().sum() # There are 2721 customers will no last_transaction_date, that's correct 

# Filing those nulls with max date of last transaction 
max_days = customers_df['days_since_last_trans'].max()
customers_df['days_since_last_trans'] =  customers_df['days_since_last_trans'].fillna(max_days)

In [8]:
# Feature Engineering - Customers Column - recency_score

In [9]:
# Recency thresholds based on our lifecycle logic
# max_days = 587
# Active = transacted within 90 days
# Declining = 90-180 days  
# Dormant = 180+ days

customers_df['recency_score'] = pd.cut(
    customers_df['days_since_last_trans'],
    bins=[-0.001, 30, 90, 180, 365, float('inf')],
    labels=[5, 4, 3, 2, 1],
    right=True
)

print(customers_df['recency_score'].value_counts().sort_index())

recency_score
5    1861
4     247
3      98
2      62
1    2732
Name: count, dtype: int64


In [10]:
# Feature Engineering - Customers columns - frequency 

In [11]:
trans_per_cust = transactions_df[transactions_df['transaction_status']=='Successful'].groupby('customer_id')['transaction_id'].count().reset_index()
trans_per_cust.columns = ['customer_id', 'total_transactions']
customers_df = customers_df.merge(trans_per_cust, on='customer_id', how='left')
customers_df['total_transactions'] = customers_df['total_transactions'].fillna(0)
customers_df['frequency_score'] = pd.cut(
 customers_df['total_transactions'],
 bins=[-0.001, 50, 100, 200, 300, float('inf')],
 labels=[1,2,3,4,5],
 right=True
)

print(customers_df['frequency_score'].value_counts().sort_index())
# and got score 1 for 3679 rows, score 2 for 375 rows, score 3 for 451 rows, score 4 for 352 rows and score 5 for 143 rows, i means we have only 495-500 customers which has good number of transactions rest 3679 customers i.e 73.58 have very less or 0 transactions (a flag for Mr. Mehta)

frequency_score
1    3679
2     375
3     451
4     352
5     143
Name: count, dtype: int64


In [12]:
# Feature Engineering - Customers columns - monetary 

In [34]:
avg_trans_per_cust = avg_trans_per_cust = transactions_df[transactions_df['transaction_status']=='Successful'].groupby('customer_id')['amount'].mean().reset_index()
avg_trans_per_cust.columns = ['customer_id', 'avg_trans_value']
customers_df = customers_df.merge(avg_trans_per_cust, on='customer_id', how='left')
customers_df['avg_trans_value'] = customers_df['avg_trans_value'].fillna(0)
customers_df['monetary_score'] = pd.cut(
 customers_df['avg_trans_value'],
 bins = [-0.001, 5401.03, 10646.76, 11590.05, 13887.93, float('inf')],
 labels=[1,2,3,4,5],
 right=True
)

print(customers_df['monetary_score'].value_counts().sort_index())

# and got score 1 for 3750 rows, score 2 for 750 rows, score 3 for 250 rows, score 4 for 200 rows and score 5 for 50 rows, i means we have 200-250 customers which has good avg transaction amount rest 3750  customers i.e 75% customers have very less or 0 transactions (a flag for Mr. Mehta)

monetary_score
1    3750
2     750
3     250
4     200
5      50
Name: count, dtype: int64


In [35]:
customers_df['recency_score'] = customers_df['recency_score'].astype(float)
customers_df['frequency_score'] = customers_df['frequency_score'].astype(float)
customers_df['monetary_score'] = customers_df['monetary_score'].astype(float)

customers_df['rfm_score'] = (
    customers_df['recency_score'] + 
    customers_df['frequency_score'] + 
    customers_df['monetary_score']
)

customers_df['rfm_segment'] = pd.cut(
    customers_df['rfm_score'],
    bins=[2, 3, 7, 11, 13, 15],
    labels=['Dormant', 'At Risk', 'Low Value', 'Medium Value', 'High Value'],
    right=True
)

print(customers_df['rfm_segment'].value_counts().sort_index())

rfm_segment
Dormant         2732
At Risk          734
Low Value       1249
Medium Value     267
High Value        18
Name: count, dtype: int64


In [36]:
# Feature Engineering - Customers column - lifecycle_stage

In [39]:
import numpy as np
# Convert the DataFrame columns to true pandas datetime format
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'])
customers_df['last_transaction_date'] = pd.to_datetime(customers_df['last_transaction_date'])

# Ensure max_date is also a pandas Timestamp
max_date = pd.to_datetime(max_transaction_date)

# Your existing code will now run flawlessly
sixty_days_ago = max_date - pd.Timedelta(days=60)
ninety_days_ago = max_date - pd.Timedelta(days=90)
one_eighty_days_ago = max_date - pd.Timedelta(days=180)

conditions = [
    (customers_df['signup_date'] >= sixty_days_ago) & (customers_df['last_transaction_date'].notna()),
    (customers_df['last_transaction_date'] >= ninety_days_ago),
    (customers_df['last_transaction_date'] >= one_eighty_days_ago) & (customers_df['last_transaction_date'] < ninety_days_ago),
]
choices = ['New', 'Active', 'Declining']
customers_df['lifecycle_stage'] = np.select(conditions, choices, default='Dormant')
print(customers_df['lifecycle_stage'].value_counts())

lifecycle_stage
Dormant      2796
Active       1978
New           127
Declining      99
Name: count, dtype: int64


In [40]:
# Feature Engineering - Customers column - inflow & outflow

In [41]:
outflow_per_cust = transactions_df[(transactions_df['transaction_status']=='Successful') & (transactions_df['debit_or_credit']=='Debit')].groupby('customer_id')['amount'].sum().reset_index()
outflow_per_cust.columns = ['customer_id', 'total_outflow_amount']
customers_df = customers_df.merge(outflow_per_cust, on='customer_id', how='left')
customers_df['total_outflow_amount'] = customers_df['total_outflow_amount'].fillna(0)

In [42]:
inflow_per_cust = transactions_df[(transactions_df['transaction_status']=='Successful') & (transactions_df['debit_or_credit']=='Credit')].groupby('customer_id')['amount'].sum().reset_index()
inflow_per_cust.columns = ['customer_id', 'total_inflow_amount']
customers_df = customers_df.merge(inflow_per_cust, on='customer_id', how='left')
customers_df['total_inflow_amount'] = customers_df['total_inflow_amount'].fillna(0)

In [43]:
customers_df['inflow_outflow_ratio'] = np.where(
    customers_df['total_outflow_amount'] == 0,
    np.nan,
    customers_df['total_inflow_amount'] / customers_df['total_outflow_amount']
)

In [50]:
conditions = [
    (customers_df['total_inflow_amount'] == 0) & (customers_df['total_outflow_amount'] == 0),
    customers_df['inflow_outflow_ratio'] > 1.1,
    customers_df['inflow_outflow_ratio'] < 0.9,
    (customers_df['inflow_outflow_ratio'] >= 0.9) & 
    (customers_df['inflow_outflow_ratio'] <= 1.1),
    (customers_df['total_inflow_amount'] > 0) & (customers_df['total_outflow_amount'] == 0),
]
choices = ['No Cash Flow', 'Increasing', 'Decreasing', 'Stable', 'Inflow Only']

customers_df['balance_trend'] = np.select(conditions, choices, default='No Cash Flow')
print(customers_df['balance_trend'].value_counts())

balance_trend
No Cash Flow    2748
Decreasing      1108
Stable           565
Inflow Only      310
Increasing       269
Name: count, dtype: int64


In [46]:
branches_df.to_sql(
    name='cleaned_branches',
    con=engine,
    if_exists='replace',
    index=False
)

10

In [53]:
customers_df.to_sql(
    name='cleaned_customers',
    con=engine,
    if_exists='replace',
    index=False
)

1000

In [54]:
accounts_df.to_sql(
    name='cleaned_accounts',
    con=engine,
    if_exists='replace',
    index=False
)

356

In [56]:
transactions_df.to_sql(
    name='cleaned_transactions',
    con=engine,
    if_exists='replace',
    index=False
)

129